In [1]:
from skillcorner.client import SkillcornerClient
from pprint import pprint
import pandas as pd
from io import BytesIO
from IPython.display import display

In [2]:
import os
USERNAME = os.environ["SC_USERNAME"]
PASSWORD = os.environ["SC_PASSWORD"]
DATA_DIR = os.environ.get("DATA_DIR", os.path.expanduser("~/Desktop/analisis_ca/scout_dashboard/data"))

client = SkillcornerClient(username=USERNAME, password=PASSWORD)
client

In [ ]:
# Retry helper for Ecuador/Championship (EFL)/Rusia -- the three leagues
# confirmed (2026-10-23/24) to intermittently return 0 rows for the SAME
# params that returned real data on an earlier run (and vice versa),
# unlike the other ~19 leagues pulled below, which reliably return data
# every single run. Traced all the way through join_scout_data.R's
# crosswalk output to Mongo before being caught -- a run with a transient
# 0-row response looked identical to "genuinely not licensed this
# season" with no way to tell them apart after the fact. This retries a
# few times with a short pause before treating an empty result as real,
# rather than bailing on the first attempt.
import time

def get_physical_with_retry(client, params, retries=3, delay_seconds=30):
    df = pd.DataFrame()
    for attempt in range(1, retries + 1):
        data = client.get_physical(params=params)
        df = pd.DataFrame(data)
        if not df.empty:
            if attempt > 1:
                print(f"    Succeeded on retry {attempt}/{retries}")
            return df
        if attempt < retries:
            print(f"    Attempt {attempt}/{retries} returned 0 rows -- retrying in {delay_seconds}s...")
            time.sleep(delay_seconds)
    print(f"    All {retries} attempts returned 0 rows -- treating as genuinely no data this run.")
    return df


In [ ]:
# PSV-99: pico por partido vs. agregado de temporada
#
# psv99 (agregado de temporada, arriba en cada celda de liga) es el valor
# que la API de SkillCorner ya devuelve al pedir group_by con "group" (sin
# "match") -- un agregado propio de SkillCorner sobre toda la temporada.
# Los scouts piden en cambio el pico real: el valor MAS ALTO de psv99 entre
# los partidos individuales del jugador en esa temporada (group_by con
# "match" en vez de "group" da una fila por partido). Un solo partido con
# un salto de GPS/tracking puede inflar ese máximo de forma poco realista,
# asi que si el valor mas alto supera al segundo mas alto por mas de
# outlier_margin km/h, se descarta como outlier y se reporta el segundo
# valor mas alto en su lugar. Con un unico partido no hay con que comparar,
# asi que ese valor se usa tal cual.
#
# Reutiliza los mismos dicts `params` que cada celda de liga ya arma para
# su pull de temporada (capturados en `all_params_used`) en vez de que este
# helper vuelva a adivinar season/competition -- evita duplicar (y
# potencialmente desincronizar) esa lógica por liga.
import numpy as np

def compute_psv99_match_peak(client, params_list, outlier_margin=1.5):
    match_rows = []
    seen = set()
    for p in params_list:
        key = (p.get("season"), p.get("competition"))
        if key in seen:
            continue
        seen.add(key)
        match_params = dict(p)
        match_params["group_by"] = "player,team,competition,season,match"
        # PSV-99 no depende de la posesión -- sin "possession" la API
        # devuelve un solo psv99 sin split ALL/TIP/OTIP por partido.
        match_params.pop("possession", None)
        data = client.get_physical(params=match_params)
        df_match = pd.DataFrame(data)
        if df_match.empty or "psv99" not in df_match.columns:
            print(f"  [psv99_match_peak] season={p.get('season')} competition={p.get('competition')}: sin filas psv99 a nivel partido")
            continue
        match_rows.append(df_match[["player_id", "psv99"]])

    if not match_rows:
        return pd.DataFrame(columns=["player_id", "psv99_match_peak"]).set_index("player_id")

    all_matches = pd.concat(match_rows, ignore_index=True)

    def _peak_with_outlier_check(values):
        top = sorted(values.dropna(), reverse=True)
        if not top:
            return np.nan
        if len(top) == 1 or (top[0] - top[1]) <= outlier_margin:
            return top[0]
        return top[1]

    peaks = all_matches.groupby("player_id")["psv99"].apply(_peak_with_outlier_check)
    return peaks.rename("psv99_match_peak").to_frame()


In [ ]:
# Shared pull helper for every league cell below -- factors out the
# "loop over seasons, pull each competition for that season, combine
# same-season competitions (rate-like metrics via max, cumulative
# counters via sum -- summing a rate like PSV99 across two editions of
# the same season silently doubled it, see the original Liga MX cell's
# comment for where this was first caught), concat seasons, compute
# psv99_match_peak, save to CSV" logic that used to be duplicated
# almost verbatim in every league cell. Added 2026-11-03 alongside the
# data-quality fallback system -- each league now calls this up to 4
# times (default/extended-seasons/tier20/tier10), so de-duplicating it
# first keeps each league cell to a handful of lines instead of
# ~120.
#
# Uses get_physical_with_retry() for every call now (previously only
# Ecuador/Championship/Rusia) -- the fallback system triples the
# number of API calls per league, so the same transient-empty-response
# flakiness those three leagues showed is now more likely to bite
# others too; retrying is strictly safer and costs nothing when a
# league doesn't need it.
#
# count_match__gte is deliberately never passed here -- the 5-match
# minimum is enforced downstream (sc_physical_acumulado_fallback.R, R
# side) on the count summed ACROSS whichever candidate sets a player's
# blend ends up using, not server-side on a single request.
from skillcornerviz.utils import skillcorner_physical_utils as p_utils

def pull_and_save_physical(client, season_plan, playing_time_gte, filename, label):
    """
    season_plan: list of (season_id, [comp_ids]) tuples.
    playing_time_gte: int, passed through to the API params.
    filename: output CSV filename (saved under DATA_DIR/physical/).
    label: short string used only in print() messages (e.g. "Liga MX default").
    Returns the saved DataFrame (empty if nothing was returned for any season).
    """
    all_params_used = []
    season_dfs = []

    for season_id, comp_ids in season_plan:
        dfs = []
        for comp_id in comp_ids:
            params = {
                "season": season_id,
                "competition": comp_id,
                "group_by": "player,team,competition,season,group",
                "possession": "all,tip,otip",
                "data_version": "3",
                "playing_time__gte": playing_time_gte,
            }
            all_params_used.append(dict(params))
            df_comp = get_physical_with_retry(client, params)
            dfs.append(df_comp)
            print(f"  [{label}] Season {season_id}, Competition {comp_id}: {len(df_comp)} rows")

        df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
        if df.empty:
            print(f"  [{label}] Season {season_id}: no rows returned, skipping.")
            continue

        if len(comp_ids) > 1:
            meta_cols = ["player_id", "player_name", "team_id", "team_name", "season", "group", "competition",
                         "season_id", "season_name", "competition_id", "competition_name"]
            rate_like_patterns = ["per_60", "per_minute", "psv", "_pct", "percentage", "ratio"]
            numeric_cols = [c for c in df.columns if c not in meta_cols and df[c].dtype in ["float64", "int64"]]
            max_cols = [c for c in numeric_cols if any(p in c.lower() for p in rate_like_patterns)]
            sum_cols = [c for c in numeric_cols if c not in max_cols]
            other_meta = [c for c in df.columns if c not in ["player_id", "player_name"] and c not in sum_cols and c not in max_cols]

            df_season = df.groupby(["player_id", "player_name"], as_index=False).agg(
                **{col: (col, "last") for col in other_meta},
                **{col: (col, "sum") for col in sum_cols},
                **{col: (col, "max") for col in max_cols}
            )
        else:
            df_season = df

        before = set(df_season.columns)
        p_utils.add_standard_metrics(df_season)
        added = sorted(set(df_season.columns) - before)
        print(f"  [{label}] Season {season_id} -- Added columns:", added)

        season_dfs.append(df_season)

    if not season_dfs:
        print(f"  [{label}] No data returned for any season -- skipping file write.")
        return pd.DataFrame()

    df_all = pd.concat(season_dfs, ignore_index=True)

    desktop_path = os.path.join(DATA_DIR, "physical")
    os.makedirs(desktop_path, exist_ok=True)
    file_path = os.path.join(desktop_path, filename)

    psv99_peaks = compute_psv99_match_peak(client, all_params_used)
    df_all = df_all.merge(psv99_peaks, on="player_id", how="left")
    print(f"  [{label}] psv99_match_peak non-null:", df_all["psv99_match_peak"].notna().sum(), "/", len(df_all))

    df_all.to_csv(file_path, index=False)
    print(f"  [{label}] Saved to:", file_path, "-- Shape:", df_all.shape)
    return df_all

In [ ]:
# ID DE LIGAS

# Liga MX: 97 y 610
# Concachampions: 159
# Argentina: 70
# Brasil: 77
# Colombia: 100
# Ecuador: 163
# Paraguay: 140
# Uruguay: 95
# MLS: 100
# La Liga: 4
# La Liga 2: 42
# Chile: 57
# Premier League: 1
# Championship: 31
# Serie A: 5
# Serie B: 41
# Bundesliga: 6
# Bundesliga 2: 40
# Ligue 1: 3
# Eredivisie: 17
# Bélgica: 16
# Portugal: 25
# Turquía: 29
# Escocia: 18
# UCL: 10
# UEL: 9
# Libertadores: 109

In [4]:
# ID DE TEMPORADAS

# 25/26: 129
# 2025: 128
# 2026: 130

# Extrapolados 2026-10-20 (no confirmados contra la API en vivo --
# ver comentarios de las celdas Championship/Rusia/Ecuador):
# 24/25: 127
# 2024: 126
# 23/24: 125


In [ ]:
# LIGA MX

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [610, 97]), (131, [610, 97])]
pull_and_save_physical(client, default_plan, 30, "ligamx_physical_standardized.csv", "LIGA MX default")

extended_plan = [(125, [610, 97]), (127, [610, 97])]
pull_and_save_physical(client, extended_plan, 30, "ligamx_physical_extended_seasons.csv", "LIGA MX extended")

pull_and_save_physical(client, default_plan, 20, "ligamx_physical_tier20.csv", "LIGA MX tier20")
pull_and_save_physical(client, default_plan, 10, "ligamx_physical_tier10.csv", "LIGA MX tier10")

In [ ]:
# ARGENTINA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [70]), (130, [70, 376])]
pull_and_save_physical(client, default_plan, 30, "arg_physical_standardized.csv", "ARGENTINA default")

extended_plan = [(124, [70]), (126, [70])]
pull_and_save_physical(client, extended_plan, 30, "arg_physical_extended_seasons.csv", "ARGENTINA extended")

pull_and_save_physical(client, default_plan, 20, "arg_physical_tier20.csv", "ARGENTINA tier20")
pull_and_save_physical(client, default_plan, 10, "arg_physical_tier10.csv", "ARGENTINA tier10")

In [ ]:
# BRASIL

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [77]), (130, [77])]
pull_and_save_physical(client, default_plan, 30, "brasil_physical_standardized.csv", "BRASIL default")

extended_plan = [(124, [77]), (126, [77])]
pull_and_save_physical(client, extended_plan, 30, "brasil_physical_extended_seasons.csv", "BRASIL extended")

pull_and_save_physical(client, default_plan, 20, "brasil_physical_tier20.csv", "BRASIL tier20")
pull_and_save_physical(client, default_plan, 10, "brasil_physical_tier10.csv", "BRASIL tier10")

In [ ]:
# COLOMBIA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [100]), (130, [100, 653])]
pull_and_save_physical(client, default_plan, 30, "col_physical_standardized.csv", "COLOMBIA default")

extended_plan = [(124, [100]), (126, [100])]
pull_and_save_physical(client, extended_plan, 30, "col_physical_extended_seasons.csv", "COLOMBIA extended")

pull_and_save_physical(client, default_plan, 20, "col_physical_tier20.csv", "COLOMBIA tier20")
pull_and_save_physical(client, default_plan, 10, "col_physical_tier10.csv", "COLOMBIA tier10")

In [ ]:
# CHILE

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [57]), (130, [57])]
pull_and_save_physical(client, default_plan, 30, "chile_physical_standardized.csv", "CHILE default")

extended_plan = [(124, [57]), (126, [57])]
pull_and_save_physical(client, extended_plan, 30, "chile_physical_extended_seasons.csv", "CHILE extended")

pull_and_save_physical(client, default_plan, 20, "chile_physical_tier20.csv", "CHILE tier20")
pull_and_save_physical(client, default_plan, 10, "chile_physical_tier10.csv", "CHILE tier10")

In [ ]:
# MLS

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [60]), (130, [60])]
pull_and_save_physical(client, default_plan, 30, "mls_physical_standardized.csv", "MLS default")

extended_plan = [(124, [60]), (126, [60])]
pull_and_save_physical(client, extended_plan, 30, "mls_physical_extended_seasons.csv", "MLS extended")

pull_and_save_physical(client, default_plan, 20, "mls_physical_tier20.csv", "MLS tier20")
pull_and_save_physical(client, default_plan, 10, "mls_physical_tier10.csv", "MLS tier10")

In [ ]:
# PREMIER LEAGUE

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [1]), (131, [1])]
pull_and_save_physical(client, default_plan, 30, "premier_physical_standardized.csv", "PREMIER LEAGUE default")

extended_plan = [(125, [1]), (127, [1])]
pull_and_save_physical(client, extended_plan, 30, "premier_physical_extended_seasons.csv", "PREMIER LEAGUE extended")

pull_and_save_physical(client, default_plan, 20, "premier_physical_tier20.csv", "PREMIER LEAGUE tier20")
pull_and_save_physical(client, default_plan, 10, "premier_physical_tier10.csv", "PREMIER LEAGUE tier10")

In [ ]:
# LA LIGA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [4]), (131, [4])]
pull_and_save_physical(client, default_plan, 30, "laliga_physical_standardized.csv", "LA LIGA default")

extended_plan = [(125, [4]), (127, [4])]
pull_and_save_physical(client, extended_plan, 30, "laliga_physical_extended_seasons.csv", "LA LIGA extended")

pull_and_save_physical(client, default_plan, 20, "laliga_physical_tier20.csv", "LA LIGA tier20")
pull_and_save_physical(client, default_plan, 10, "laliga_physical_tier10.csv", "LA LIGA tier10")

In [ ]:
# LA LIGA 2

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [42]), (131, [42])]
pull_and_save_physical(client, default_plan, 30, "laliga2_physical_standardized.csv", "LA LIGA 2 default")

extended_plan = [(125, [42]), (127, [42])]
pull_and_save_physical(client, extended_plan, 30, "laliga2_physical_extended_seasons.csv", "LA LIGA 2 extended")

pull_and_save_physical(client, default_plan, 20, "laliga2_physical_tier20.csv", "LA LIGA 2 tier20")
pull_and_save_physical(client, default_plan, 10, "laliga2_physical_tier10.csv", "LA LIGA 2 tier10")

In [ ]:
# SERIE A

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [5]), (131, [5])]
pull_and_save_physical(client, default_plan, 30, "seriea_physical_standardized.csv", "SERIE A default")

extended_plan = [(125, [5]), (127, [5])]
pull_and_save_physical(client, extended_plan, 30, "seriea_physical_extended_seasons.csv", "SERIE A extended")

pull_and_save_physical(client, default_plan, 20, "seriea_physical_tier20.csv", "SERIE A tier20")
pull_and_save_physical(client, default_plan, 10, "seriea_physical_tier10.csv", "SERIE A tier10")

In [ ]:
# TURQUÍA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [29]), (131, [29])]
pull_and_save_physical(client, default_plan, 30, "tur_physical_standardized.csv", "TURQUÍA default")

extended_plan = [(125, [29]), (127, [29])]
pull_and_save_physical(client, extended_plan, 30, "tur_physical_extended_seasons.csv", "TURQUÍA extended")

pull_and_save_physical(client, default_plan, 20, "tur_physical_tier20.csv", "TURQUÍA tier20")
pull_and_save_physical(client, default_plan, 10, "tur_physical_tier10.csv", "TURQUÍA tier10")

In [ ]:
# UEFA CHAMPIONS LEAGUE

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [10]), (131, [10])]
pull_and_save_physical(client, default_plan, 30, "ucl_physical_standardized.csv", "UEFA CHAMPIONS LEAGUE default")

extended_plan = [(125, [10]), (127, [10])]
pull_and_save_physical(client, extended_plan, 30, "ucl_physical_extended_seasons.csv", "UEFA CHAMPIONS LEAGUE extended")

pull_and_save_physical(client, default_plan, 20, "ucl_physical_tier20.csv", "UEFA CHAMPIONS LEAGUE tier20")
pull_and_save_physical(client, default_plan, 10, "ucl_physical_tier10.csv", "UEFA CHAMPIONS LEAGUE tier10")

In [ ]:
# UEFA EUROPA LEAGUE

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [9]), (131, [9])]
pull_and_save_physical(client, default_plan, 30, "uel_physical_standardized.csv", "UEFA EUROPA LEAGUE default")

extended_plan = [(125, [9]), (127, [9])]
pull_and_save_physical(client, extended_plan, 30, "uel_physical_extended_seasons.csv", "UEFA EUROPA LEAGUE extended")

pull_and_save_physical(client, default_plan, 20, "uel_physical_tier20.csv", "UEFA EUROPA LEAGUE tier20")
pull_and_save_physical(client, default_plan, 10, "uel_physical_tier10.csv", "UEFA EUROPA LEAGUE tier10")

In [ ]:
# PARAGUAY

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [140]), (130, [140])]
pull_and_save_physical(client, default_plan, 30, "paraguay_physical_standardized.csv", "PARAGUAY default")

extended_plan = [(124, [140]), (126, [140])]
pull_and_save_physical(client, extended_plan, 30, "paraguay_physical_extended_seasons.csv", "PARAGUAY extended")

pull_and_save_physical(client, default_plan, 20, "paraguay_physical_tier20.csv", "PARAGUAY tier20")
pull_and_save_physical(client, default_plan, 10, "paraguay_physical_tier10.csv", "PARAGUAY tier10")

In [ ]:
# URUGUAY

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [95]), (130, [95, 655])]
pull_and_save_physical(client, default_plan, 30, "uruguay_physical_standardized.csv", "URUGUAY default")

extended_plan = [(124, [95]), (126, [95])]
pull_and_save_physical(client, extended_plan, 30, "uruguay_physical_extended_seasons.csv", "URUGUAY extended")

pull_and_save_physical(client, default_plan, 20, "uruguay_physical_tier20.csv", "URUGUAY tier20")
pull_and_save_physical(client, default_plan, 10, "uruguay_physical_tier10.csv", "URUGUAY tier10")

In [ ]:
# PERU

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(128, [139]), (130, [139, 652])]
pull_and_save_physical(client, default_plan, 30, "peru_physical_standardized.csv", "PERU default")

extended_plan = [(124, [139]), (126, [139])]
pull_and_save_physical(client, extended_plan, 30, "peru_physical_extended_seasons.csv", "PERU extended")

pull_and_save_physical(client, default_plan, 20, "peru_physical_tier20.csv", "PERU tier20")
pull_and_save_physical(client, default_plan, 10, "peru_physical_tier10.csv", "PERU tier10")

In [ ]:
# ECUADOR

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(126, [163]), (128, [163]), (130, [163])]
pull_and_save_physical(client, default_plan, 30, "ecuador_physical_standardized.csv", "ECUADOR default")

# Already pulls >=4 seasons by default -- no separate extended-seasons file needed.

pull_and_save_physical(client, default_plan, 20, "ecuador_physical_tier20.csv", "ECUADOR tier20")
pull_and_save_physical(client, default_plan, 10, "ecuador_physical_tier10.csv", "ECUADOR tier10")

In [ ]:
# CHAMPIONSHIP (ENG)

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(125, [31]), (127, [31]), (129, [31]), (131, [31])]
pull_and_save_physical(client, default_plan, 30, "championship_physical_standardized.csv", "CHAMPIONSHIP (ENG) default")

# Already pulls >=4 seasons by default -- no separate extended-seasons file needed.

pull_and_save_physical(client, default_plan, 20, "championship_physical_tier20.csv", "CHAMPIONSHIP (ENG) tier20")
pull_and_save_physical(client, default_plan, 10, "championship_physical_tier10.csv", "CHAMPIONSHIP (ENG) tier10")

In [ ]:
# BUNDESLIGA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [6]), (131, [6])]
pull_and_save_physical(client, default_plan, 30, "bundesliga_physical_standardized.csv", "BUNDESLIGA default")

extended_plan = [(125, [6]), (127, [6])]
pull_and_save_physical(client, extended_plan, 30, "bundesliga_physical_extended_seasons.csv", "BUNDESLIGA extended")

pull_and_save_physical(client, default_plan, 20, "bundesliga_physical_tier20.csv", "BUNDESLIGA tier20")
pull_and_save_physical(client, default_plan, 10, "bundesliga_physical_tier10.csv", "BUNDESLIGA tier10")

In [ ]:
# 2. BUNDESLIGA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [40]), (131, [40])]
pull_and_save_physical(client, default_plan, 30, "bundesliga2_physical_standardized.csv", "2. BUNDESLIGA default")

extended_plan = [(125, [40]), (127, [40])]
pull_and_save_physical(client, extended_plan, 30, "bundesliga2_physical_extended_seasons.csv", "2. BUNDESLIGA extended")

pull_and_save_physical(client, default_plan, 20, "bundesliga2_physical_tier20.csv", "2. BUNDESLIGA tier20")
pull_and_save_physical(client, default_plan, 10, "bundesliga2_physical_tier10.csv", "2. BUNDESLIGA tier10")

In [ ]:
# LIGUE 1

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [3]), (131, [3])]
pull_and_save_physical(client, default_plan, 30, "ligue1_physical_standardized.csv", "LIGUE 1 default")

extended_plan = [(125, [3]), (127, [3])]
pull_and_save_physical(client, extended_plan, 30, "ligue1_physical_extended_seasons.csv", "LIGUE 1 extended")

pull_and_save_physical(client, default_plan, 20, "ligue1_physical_tier20.csv", "LIGUE 1 tier20")
pull_and_save_physical(client, default_plan, 10, "ligue1_physical_tier10.csv", "LIGUE 1 tier10")

In [ ]:
# EREDIVISIE

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [17]), (131, [17])]
pull_and_save_physical(client, default_plan, 30, "eredivisie_physical_standardized.csv", "EREDIVISIE default")

extended_plan = [(125, [17]), (127, [17])]
pull_and_save_physical(client, extended_plan, 30, "eredivisie_physical_extended_seasons.csv", "EREDIVISIE extended")

pull_and_save_physical(client, default_plan, 20, "eredivisie_physical_tier20.csv", "EREDIVISIE tier20")
pull_and_save_physical(client, default_plan, 10, "eredivisie_physical_tier10.csv", "EREDIVISIE tier10")

In [ ]:
# BÉLGICA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [16]), (131, [16])]
pull_and_save_physical(client, default_plan, 30, "belgica_physical_standardized.csv", "BÉLGICA default")

extended_plan = [(125, [16]), (127, [16])]
pull_and_save_physical(client, extended_plan, 30, "belgica_physical_extended_seasons.csv", "BÉLGICA extended")

pull_and_save_physical(client, default_plan, 20, "belgica_physical_tier20.csv", "BÉLGICA tier20")
pull_and_save_physical(client, default_plan, 10, "belgica_physical_tier10.csv", "BÉLGICA tier10")

In [ ]:
# PORTUGAL

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [25]), (131, [25])]
pull_and_save_physical(client, default_plan, 30, "portugal_physical_standardized.csv", "PORTUGAL default")

extended_plan = [(125, [25]), (127, [25])]
pull_and_save_physical(client, extended_plan, 30, "portugal_physical_extended_seasons.csv", "PORTUGAL extended")

pull_and_save_physical(client, default_plan, 20, "portugal_physical_tier20.csv", "PORTUGAL tier20")
pull_and_save_physical(client, default_plan, 10, "portugal_physical_tier10.csv", "PORTUGAL tier10")

In [ ]:
# ESCOCIA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(129, [18]), (131, [18])]
pull_and_save_physical(client, default_plan, 30, "escocia_physical_standardized.csv", "ESCOCIA default")

extended_plan = [(125, [18]), (127, [18])]
pull_and_save_physical(client, extended_plan, 30, "escocia_physical_extended_seasons.csv", "ESCOCIA extended")

pull_and_save_physical(client, default_plan, 20, "escocia_physical_tier20.csv", "ESCOCIA tier20")
pull_and_save_physical(client, default_plan, 10, "escocia_physical_tier10.csv", "ESCOCIA tier10")

In [ ]:
# RUSIA

client = SkillcornerClient(username=USERNAME, password=PASSWORD)

default_plan = [(125, [27]), (127, [27]), (129, [27]), (131, [27])]
pull_and_save_physical(client, default_plan, 30, "rusia_physical_standardized.csv", "RUSIA default")

# Already pulls >=4 seasons by default -- no separate extended-seasons file needed.

pull_and_save_physical(client, default_plan, 20, "rusia_physical_tier20.csv", "RUSIA tier20")
pull_and_save_physical(client, default_plan, 10, "rusia_physical_tier10.csv", "RUSIA tier10")